In [1]:
%load_ext autoreload
%autoreload 2
import cftime
import dask
import glob
from MOCutils import popmoc
import numpy as np  
import os
import time
import xarray as xr 
from xhistogram.xarray import histogram

In [2]:
from dask.distributed import wait
dask.__version__

'2024.7.0'

In [3]:
# Close out Dask Cluster and release workers:
#cluster.close()
#client.close()

In [4]:
# TODO: optimize dask resources

def get_ClusterClient():
    import dask
    from dask_jobqueue import PBSCluster
    from dask.distributed import Client
    cluster = PBSCluster(
        cores=1,
        memory='20GB',
        processes=1,
        queue='casper',
        resource_spec='select=1:ncpus=1:mem=20GB',
        project='NCGD0011',
        walltime='12:00:00',
        interface='ext',)

    dask.config.set({
        'distributed.dashboard.link':
        'https://jupyterhub.hpc.ucar.edu/stable/user/{USER}/proxy/{port}/status'
    })
    client = Client(cluster)
    return cluster, client

cluster, client = get_ClusterClient()
cluster.scale(72) 

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask_jobqueue/pbs.py:82: FutureWarning: project has been renamed to account as this kwarg was used wit -A option. You are still using it (please also check config files). If you did not set account yet, project will be respected for now, but it will be removed in a future release. If you already set account, project is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.1

In [5]:
# shift VVEL to 3121 position
def shiftVVEL(v):
    v_w = v.roll(nlon=1,roll_coords=False)
    v = 0.5*(v+v_w)
    return v.drop_vars(['ULONG','TLAT'])

# Grid metrics

In [6]:
fgrd = '/glade/work/fredc/cesm/grid/POP/POP_tx0.1v2_grid.nc'
ds_grid = xr.open_dataset(fgrd)

In [7]:
tlon = ds_grid.TLONG.drop_vars(['ULONG','ULAT'])
tlat = ds_grid.TLAT.drop_vars(['ULONG','ULAT'])
ulon = ds_grid.ULONG.drop_vars(['TLONG','TLAT'])
ulat = ds_grid.ULAT.drop_vars(['TLONG','TLAT'])

In [8]:
fmoc = 'MOCutils/moc_template.nc'
ds_moctemp = xr.open_dataset(fmoc)

In [9]:
## Define the MOC region mask:
rmask = ds_grid.REGION_MASK.drop_vars(['ULONG','ULAT'])
rmaskglob = xr.where((rmask>0),1,0)
rmaskatl = xr.where((rmask>=6) & (rmask<=11),1,0)
rmaskmoc = xr.concat([rmaskglob,rmaskatl],dim=ds_moctemp.transport_regions)

In [10]:
# determine j=index of Atlantic region southern boundary
tmp = rmaskmoc.isel(transport_reg=1).sum('nlon')
atl_j = 0
j = 0
while (atl_j==0):
    if (tmp.isel(nlat=j).data>0):
        atl_j = j
    j += 1
atl_j = atl_j - 1

# Loop over files and compute MOC 

In [11]:
fdir = '/glade/derecho/scratch/fredc/archive/'
case = 'b.e13.B1950C5.ne120_t12.cesm-ihesp-hires1.0.46.4xCO2.test'

fpath = '{}/{}/ocn/hist/{}.pop.h.0*.nc'.format(fdir,case,case)
flst = sorted(glob.glob(fpath))

In [12]:
nf = len(flst)

In [13]:
for n in range(nf):

    fin = flst[n]

    ds = xr.open_dataset(fin, chunks={'time':1,'nlon':100})
    ds['z_t'] = ds_grid.z_t
    ds['z_w'] = ds_grid.z_w
    ds['z_w_top'] = ds_grid.z_w_top
    ds['z_w_bot'] = ds_grid.z_w_bot
    ds['z_t_150m'] = ds_grid.z_t_150m

    v_e = ds['VVEL']
    v_e = v_e.where(v_e<1.e30,0)
    v_e = v_e/100.
    v_e = shiftVVEL(v_e)

    w_e = ds['WVEL'].drop_vars(['ULONG','ULAT'])
    w_e = w_e.where(w_e<1.e30, 0)
    w_e = w_e/100.

    # grid-oriented volume fluxes in m^3/s
    tarea = ds_grid['TAREA'].drop_vars(['ULONG','ULAT']).astype(w_e.dtype)
    w_e = w_e*tarea
    dxdz = (ds_grid['HTN'].drop_vars(['ULONG','TLAT'])*ds_grid['dz']).astype(v_e.dtype)
    v_e = v_e*dxdz
    
    #add an extra level corresponding to zero values at ocean floor. (requires rechunking)
    w_e_bot = xr.zeros_like(w_e.isel(z_w_top=-1))
    w_e_bot['z_w_top'] = ds_grid.z_w_bot[-1].data
    wflux = xr.concat([w_e,w_e_bot],dim='z_w_top').rename({'z_w_top':'moc_z'}).chunk({'moc_z':63})

    v_e_bot = xr.zeros_like(v_e.isel(z_t=-1))
    v_e_bot['z_t'] = ds_grid.z_w_bot[-1].data
    vflux = xr.concat([v_e,v_e_bot],dim='z_t').chunk({'z_t':63})
    
    wflux = wflux.assign_coords({'TLONG':tlon, 'TLAT':tlat})
    vflux = vflux.assign_coords({'TLONG':tlon, 'ULAT':tlat})
    
    #Compute MOC
    MOC = popmoc.compute_MOC(wflux,rmaskmoc,ds_moctemp.lat_aux_grid)
    MOC = MOC.load()
    
    # add vflux at southern boundary of Atlantic domain
    tmp = vflux*(rmaskmoc.shift(nlat=-1))
    tmp = tmp.isel(nlat=atl_j,transport_reg=1).sum('nlon').rename({'z_t':'moc_z'})
    mocatl_s = -tmp.sortby('moc_z',ascending=False).cumsum('moc_z').sortby('moc_z',ascending=True)/1.e6
    mocatl_s['moc_z'] = MOC['moc_z']
    mocatl_s = mocatl_s.load()
    MOC[{'transport_reg':1}] = MOC[{'transport_reg':1}] + mocatl_s
    
    #Write to netcdf
    dso = MOC.to_dataset(name='MOC')
    
    outdir = os.path.dirname(fin)
    fout = os.path.split(fin)[-1]
    fout = fout.split('.')
    fout.insert(11, 'MOCz')
    fout = '.'.join(fout)
    fout = os.path.join(outdir,fout)
    
    dso.to_netcdf(fout, unlimited_dims='time')

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:3245: UserWarning: Sending large graph of size 230.76 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:3245: UserWarning: Sending large graph of size 2.12 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:3245: UserWarning: Sending large graph of size 230.76 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:3245: UserWarning: Sending large graph of size 2.12 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  war